In [27]:
import os
from dotenv import load_dotenv

# 1. Le preguntamos a Python en qué carpeta está parado
carpeta_actual = os.getcwd()
print(f"Python está buscando en esta carpeta: {carpeta_actual}")

# 2. Forzamos a que lea el archivo y sobreescriba la memoria
load_dotenv(override=True)

# 3. Intentamos leer de nuevo
api_key = os.getenv("LLM_API_KEY")

if api_key:
    print(" API Key cargada correctamente.")
else:
    print(" API NO ENCONTRADA. Fijarse si la carpeta de arriba es donde guardaste el .env, o si lo guardaste bien al archivo")

Python está buscando en esta carpeta: c:\Users\Santi\Desktop\Proyecto_integrador_bases_de_datos\IAJupyter
 API Key cargada correctamente.


In [28]:
import os
import mysql.connector
from dotenv import load_dotenv

# Cargamos las variables (por si acaso)
load_dotenv()

try:
    # 1. Intentamos establecer la conexión usando los datos del .env
    conexion = mysql.connector.connect(
        host=os.getenv("DB_HOST"),
        user=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        database=os.getenv("DB_NAME")
    )
    
    if conexion.is_connected():
        print("¡Conexión exitosa a la base de datos!")
        
        # 2. Creamos un 'cursor' (es el mensajero que lleva el SQL y trae los datos)
        cursor = conexion.cursor()
        
        # 3. Probamos ejecutar una consulta súper básica
        cursor.execute("SHOW TABLES;")
        
        # 4. Obtenemos los resultados y los imprimimos
        tablas = cursor.fetchall()
        print("\nTablas y vistas encontradas en tu base de datos:")
        for tabla in tablas:
            print(f"{tabla[0]}")
            
        # Cerramos el cursor
        cursor.close()

except mysql.connector.Error as error:
    # Si hay un error (clave mal, motor apagado), nos avisa acá
    print(f"❌ Error al conectar a MySQL: {error}")

finally:
    # 5. Siempre es buena práctica cerrar la conexión al terminar
    if 'conexion' in locals() and conexion.is_connected():
        conexion.close()
        print("\n Conexión cerrada correctamente.")

¡Conexión exitosa a la base de datos!

Tablas y vistas encontradas en tu base de datos:
auditoria_prestamos
autor
ejemplar
genero
libro
libro_autor
libro_genero
prestamo
sancion
socio
vw_libros_disponibles
vw_prestamos_activos
vw_socios_sancionados

 Conexión cerrada correctamente.


In [12]:
import os
from google import genai
from dotenv import load_dotenv

# Cargamos la clave del .env
load_dotenv()
api_key = os.getenv("LLM_API_KEY")

# 1. Inicializamos el cliente con la nueva librería
client = genai.Client(api_key=api_key)

try:
    print("Enviando mensaje a la IA con la nueva librería")
    
    # 2. Le mandamos el mensaje usando el modelo más reciente (gemini-2.5-flash)
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents='Hola! Vamos a trabajar armando consultas SQL para una biblioteca. Respondeme en una sola oración corta si estás listo.'
    )
    
    # 3. Imprimimos lo que nos contestó
    print("\n ¡Conexión exitosa con la IA!")
    print(f"Gemini dice: {response.text}")

except Exception as e:
    print(f"\n Error al conectar con la IA: {e}")

Enviando mensaje a la IA con la nueva librería

 ¡Conexión exitosa con la IA!
Gemini dice: ¡Listo!


In [2]:
import os
import mysql.connector
from google import genai
from dotenv import load_dotenv
import pandas as pd
from google.genai import types
import warnings
from IPython.display import display

# Desactivamos el aviso rojo de Pandas
warnings.filterwarnings('ignore', category=UserWarning)

# Cargamos variables de entorno
load_dotenv()
api_key = os.getenv("LLM_API_KEY")
client = genai.Client(api_key=api_key)

def ejecutar_consulta_ia(pregunta_usuario):
    print(f"🤔 Procesando pregunta: '{pregunta_usuario}'...\n")
    
    # 1. Esquema limpio y preciso de tu base de datos
    esquema_base_datos = """
    Tablas MySQL disponibles:
    - GENERO (id_genero, nombre, descripcion)
    - AUTOR (id_autor, nombre, apellido, nacionalidad)
    - LIBRO (isbn, titulo, anio_publicacion, stock_total, stock_disponible)
    - SOCIO (id_socio, dni, nombre, apellido, email, fecha_alta, estado)
    - LIBRO_AUTOR (isbn, id_autor)
    - LIBRO_GENERO (isbn, id_genero)
    - EJEMPLAR (id_ejemplar, isbn, nro_ejemplar, estado_fisico)
    - SANCION (id_sancion, id_socio, tipo, fecha_inicio, fecha_fin, motivo)
    - PRESTAMO (id_prestamo, id_socio, id_ejemplar, fecha_prestamo, fecha_vencimiento, fecha_devolucion, estado)

    Vistas disponibles (usar preferentemente si aplica a la pregunta):
    - vw_prestamos_activos (id_prestamo, nombre_socio, email_socio, titulo_libro, fecha_vencimiento, dias_de_mora)
    - vw_libros_disponibles (isbn, titulo, autor, stock_disponible)
    - vw_socios_sancionados (id_socio, nombre_socio, estado_actual, total_sanciones_historicas)
    """
    
    # 2. Instrucciones directas e inquebrantables
    system_instruction = """
    Sos un traductor de lenguaje natural a SQL para MySQL. Tu única salida debe ser código SQL ejecutable.
    Cero explicaciones, cero saludos, cero formato Markdown (no uses ```sql ni ```).
    
    Reglas relacionales estrictas:
    1. LIBRO y AUTOR se unen OBLIGATORIAMENTE pasando por la tabla intermedia LIBRO_AUTOR.
    2. LIBRO y GENERO se unen OBLIGATORIAMENTE pasando por la tabla intermedia LIBRO_GENERO.
    3. En AUTOR y SOCIO, para buscar por nombre y apellido a la vez, usa: CONCAT(nombre, ' ', apellido).
    """
    
    # 3. Llamada a la IA y ejecución
    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=f"{esquema_base_datos}\nPregunta del usuario: {pregunta_usuario}",
            config=types.GenerateContentConfig(
                system_instruction=system_instruction,
                temperature=0.1
            )
        )
        
        sql_query = response.text.strip()
        
        # Limpieza de seguridad por si manda bloques markdown
        if sql_query.startswith("```sql"): 
            sql_query = sql_query[6:]
        if sql_query.startswith("```"): 
            sql_query = sql_query[3:]
        if sql_query.endswith("```"): 
            sql_query = sql_query[:-3]
            
        sql_query = sql_query.strip()
            
        print(f"🤖 SQL generado por la IA:\n{sql_query}\n")
        
        # Conexión a la BD
        conexion = mysql.connector.connect(
            host=os.getenv("DB_HOST"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
            database=os.getenv("DB_NAME")
        )
        
        if conexion.is_connected():
            df = pd.read_sql(sql_query, con=conexion)
            return df
            
    except Exception as e:
        print(f"❌ Ocurrió un error: {e}")
        return None
        
    finally:
        if 'conexion' in locals() and conexion.is_connected():
            conexion.close()

In [15]:
pregunta = "¿Cuáles son todos los libros que pidio, actualmente tiene o devolvio el socio dni 11111111?"
resultado = ejecutar_consulta_ia(pregunta)

# Evaluamos el resultado de forma más clara
if resultado is None:
    print("❌ Hubo un error al ejecutar la consulta.")
elif resultado.empty:
    print("📭 La consulta se ejecutó bien, pero devolvió 0 resultados (no hay datos que coincidan).")
else:
    print("📊 Resultados obtenidos:")
    display(resultado)

🤔 Procesando pregunta: '¿Cuáles son todos los libros que pidio, actualmente tiene o devolvio el socio dni 11111111?'...

🤖 SQL generado por la IA:
SELECT DISTINCT L.titulo
FROM SOCIO S
JOIN PRESTAMO P ON S.id_socio = P.id_socio
JOIN EJEMPLAR E ON P.id_ejemplar = E.id_ejemplar
JOIN LIBRO L ON E.isbn = L.isbn
WHERE S.dni = '11111111';

📊 Resultados obtenidos:


,titulo
0,Ficciones
1,Fundación


In [11]:
import os
import mysql.connector
from google import genai
from dotenv import load_dotenv
import pandas as pd
from google.genai import types

# Cargamos variables (asumiendo que ya están en el entorno, pero lo aseguramos)
load_dotenv()
api_key = os.getenv("LLM_API_KEY")
client = genai.Client(api_key=api_key)

def obtener_recomendaciones_socio(identificador_socio, tipo_busqueda='id'):
    print(f"Buscando historial y armando recomendaciones para el socio ({tipo_busqueda}): {identificador_socio}...\n")
    
    conexion = None
    try:
        conexion = mysql.connector.connect(
            host=os.getenv("DB_HOST"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
            database=os.getenv("DB_NAME")
        )
        cursor = conexion.cursor(dictionary=True)
        
        # Buscar al socio explícitamente por ID o por DNI para evitar choques
        if tipo_busqueda == 'dni':
            query_socio = "SELECT id_socio, nombre, apellido FROM SOCIO WHERE dni = %s"
        else:
            query_socio = "SELECT id_socio, nombre, apellido FROM SOCIO WHERE id_socio = %s"
            
        cursor.execute(query_socio, (str(identificador_socio),))
        socio = cursor.fetchone()
        
        if not socio:
            print(f"No se encontró ningún socio con ese {tipo_busqueda.upper()}.")
            return
            
        id_socio = socio['id_socio']
        nombre_completo = f"{socio['nombre']} {socio['apellido']}"
        
        # Buscar qué leyó el socio (Historial)
        query_historial = """
            SELECT l.titulo, 
                   GROUP_CONCAT(DISTINCT g.nombre SEPARATOR ', ') AS generos,
                   GROUP_CONCAT(DISTINCT CONCAT(a.nombre, ' ', a.apellido) SEPARATOR ', ') AS autores
            FROM PRESTAMO p
            JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar
            JOIN LIBRO l ON e.isbn = l.isbn
            LEFT JOIN LIBRO_GENERO lg ON l.isbn = lg.isbn
            LEFT JOIN GENERO g ON lg.id_genero = g.id_genero
            LEFT JOIN LIBRO_AUTOR la ON l.isbn = la.isbn
            LEFT JOIN AUTOR a ON la.id_autor = a.id_autor
            WHERE p.id_socio = %s
            GROUP BY l.isbn, l.titulo;
        """
        cursor.execute(query_historial, (id_socio,))
        historial = cursor.fetchall()
        
        # Buscar libros disponibles que el socio NUNCA haya llevado
        query_candidatos = """
            SELECT l.titulo, 
                   GROUP_CONCAT(DISTINCT g.nombre SEPARATOR ', ') AS generos,
                   GROUP_CONCAT(DISTINCT CONCAT(a.nombre, ' ', a.apellido) SEPARATOR ', ') AS autores
            FROM LIBRO l
            LEFT JOIN LIBRO_GENERO lg ON l.isbn = lg.isbn
            LEFT JOIN GENERO g ON lg.id_genero = g.id_genero
            LEFT JOIN LIBRO_AUTOR la ON l.isbn = la.isbn
            LEFT JOIN AUTOR a ON la.id_autor = a.id_autor
            WHERE l.stock_disponible > 0 
              AND l.isbn NOT IN (
                  SELECT e.isbn FROM PRESTAMO p 
                  JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar 
                  WHERE p.id_socio = %s
              )
            GROUP BY l.isbn, l.titulo;
        """
        cursor.execute(query_candidatos, (id_socio,))
        candidatos = cursor.fetchall()
        
        if not candidatos:
            print("No hay libros disponibles en este momento para recomendar.")
            return

        # Preparamos la información para la IA
        texto_historial = "\n".join([f"- {h['titulo']} (Géneros: {h['generos']} | Autores: {h['autores']})" for h in historial]) if historial else "El socio es nuevo, no tiene historial de lectura."
        texto_candidatos = "\n".join([f"- {c['titulo']} (Géneros: {c['generos']} | Autores: {c['autores']})" for c in candidatos])
        
        prompt_ia = f"""
        Actúa como un bibliotecario experto. 
        Dentro de tu actuacion no respondas acciones, de tipo: *te sonrie*, *te guiña un ojo*, *te da un abrazo*, etc.
        Tienes al socio '{nombre_completo}'. 
        
        Su historial de lectura es:
        {texto_historial}
        
        Libros actualmente disponibles en la biblioteca:
        {texto_candidatos}
        
        Tu tarea:
        Elige máximo 3 libros de los "disponibles" que coincidan con sus gustos (basado en géneros o autores que ya leyó). Si es un socio nuevo, recomiéndale los 3 mejores libros en general.
        
        Para cada recomendación, debes devolver EXACTAMENTE este formato:
        TÍTULO: [Título del libro]
        PORTADA: [Genera una breve y vívida descripción visual de cómo te imaginas la portada de este libro]
        JUSTIFICACIÓN: [Escribe una justificación personalizada cruzando el libro sugerido con lo que el socio ya leyó]
        ---
        """
        
        # Enviamos a Gemini
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt_ia,
            config=types.GenerateContentConfig(
                temperature=0.7 # Un poco más de temperatura para que sea creativo con la redacción y portadas
            )
        )
        
        print(f"RECOMENDACIONES PARA {nombre_completo.upper()}")
        print("="*50)
        print(response.text)
        
    except Exception as e:
        print(f"Ocurrió un error: {e}")
    finally:
        if conexion and conexion.is_connected():
            cursor.close()
            conexion.close()




def recomendacion_lectores_afines(identificador_socio, tipo_busqueda='id'):
    print(f"Buscando recomendaciones basadas en lectores afines para el socio ({tipo_busqueda}): {identificador_socio}...\n")
    
    conexion = None
    try:
        conexion = mysql.connector.connect(
            host=os.getenv("DB_HOST"),
            user=os.getenv("DB_USER"),
            password=os.getenv("DB_PASSWORD"),
            database=os.getenv("DB_NAME")
        )
        cursor = conexion.cursor(dictionary=True)
        
        # 1. Obtener ID del socio
        query_socio = f"SELECT id_socio, nombre, apellido FROM SOCIO WHERE {'dni' if tipo_busqueda == 'dni' else 'id_socio'} = %s"
        cursor.execute(query_socio, (str(identificador_socio),))
        socio = cursor.fetchone()
        
        if not socio:
            print("Socio no encontrado.")
            return
            
        id_socio = socio['id_socio']
        nombre_completo = f"{socio['nombre']} {socio['apellido']}"

        # 2. Buscar Lectores Afines (Comparten géneros en préstamos recientes)
        # La afinidad se mide por cuántos géneros en común leyeron.
        query_afines = """
            SELECT p.id_socio, COUNT(DISTINCT lg.id_genero) as afinidad
            FROM PRESTAMO p
            JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar
            JOIN LIBRO_GENERO lg ON e.isbn = lg.isbn
            WHERE lg.id_genero IN (
                -- Géneros que leyó el socio actual
                SELECT DISTINCT lg2.id_genero 
                FROM PRESTAMO p2 
                JOIN EJEMPLAR e2 ON p2.id_ejemplar = e2.id_ejemplar 
                JOIN LIBRO_GENERO lg2 ON e2.isbn = lg2.isbn 
                WHERE p2.id_socio = %s
            )
            AND p.id_socio != %s
            GROUP BY p.id_socio
            ORDER BY afinidad DESC
            LIMIT 5;
        """
        cursor.execute(query_afines, (id_socio, id_socio))
        afines = cursor.fetchall()
        
        if not afines:
            print("No hay suficientes datos para hacer una recomendación colaborativa. Este usuario tiene gustos muy únicos o no leyó lo suficiente.")
            return
            
        id_afines = [str(a['id_socio']) for a in afines]
        
        # 3. Buscar libros que los "Afines" leyeron, pero que nuestro socio NO leyó, y que tengan stock
        format_strings = ','.join(['%s'] * len(id_afines))
        query_libros_colab = f"""
            SELECT l.titulo, 
                   GROUP_CONCAT(DISTINCT g.nombre SEPARATOR ', ') AS generos,
                   GROUP_CONCAT(DISTINCT CONCAT(a.nombre, ' ', a.apellido) SEPARATOR ', ') AS autores
            FROM PRESTAMO p
            JOIN EJEMPLAR e ON p.id_ejemplar = e.id_ejemplar
            JOIN LIBRO l ON e.isbn = l.isbn
            LEFT JOIN LIBRO_GENERO lg ON l.isbn = lg.isbn
            LEFT JOIN GENERO g ON lg.id_genero = g.id_genero
            LEFT JOIN LIBRO_AUTOR la ON l.isbn = la.isbn
            LEFT JOIN AUTOR a ON la.id_autor = a.id_autor
            WHERE p.id_socio IN ({format_strings})
            AND l.isbn NOT IN (
                -- Libros ya leídos por nuestro socio
                SELECT e2.isbn FROM PRESTAMO p2 
                JOIN EJEMPLAR e2 ON p2.id_ejemplar = e2.id_ejemplar 
                WHERE p2.id_socio = %s
            )
            AND l.stock_disponible > 0
            GROUP BY l.isbn, l.titulo
            LIMIT 10;
        """
        
        # Pasamos los IDs de los afines + el ID del socio actual para excluir
        params = tuple(id_afines) + (id_socio,)
        cursor.execute(query_libros_colab, params)
        candidatos_colab = cursor.fetchall()
        
        if not candidatos_colab:
            print("Los lectores afines leyeron exactamente los mismos libros que este socio. No hay novedades para recomendar.")
            return

        # 4. Formatear para Gemini
        texto_candidatos = "\n".join([f"- {c['titulo']} (Géneros: {c['generos']} | Autores: {c['autores']})" for c in candidatos_colab])
        
        prompt_ia = f"""
        Actúa como un bibliotecario experto. Estás hablando con el socio '{nombre_completo}'.
        Dentro de tu actuacion no respondas acciones, de tipo: *te sonrie*, *te guiña un ojo*, *te da un abrazo*, etc.
        Hemos hecho un análisis de Filtro Colaborativo en nuestra base de datos y descubrimos a un grupo de socios con gustos literarios similares a los de él/ella. 
        Estos "lectores afines" han leído y disfrutado los siguientes libros, los cuales '{nombre_completo}' AÚN NO HA LEÍDO:
        
        {texto_candidatos}
        
        Tu tarea:
        Elige 2 libros de esa lista y recomiéndaselos. 
        En tu justificación, DEBES mencionar que esta es una recomendación basada en lectores con gustos similares.
        
        Formato requerido:
        RECOMENDACIÓN DE LECTORES AFINES
        TÍTULO: [TÍTULO]
        JUSTIFICACIÓN: [Tu justificación basada en otros lectores]
        ---
        """
        
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt_ia,
            config=types.GenerateContentConfig(temperature=0.7)
        )
        
        print(f"RECOMENDACIONES DE LECTORES AFINES PARA {nombre_completo.upper()}")
        print("="*50)
        print(response.text)

    except Exception as e:
        print(f"Ocurrió un error: {e}")
    finally:
        if conexion and conexion.is_connected():
            cursor.close()
            conexion.close()
      

In [ ]:
obtener_recomendaciones_socio(11111111,tipo_busqueda='dni') 
recomendacion_lectores_afines(11111111, tipo_busqueda='dni')


Buscando historial y armando recomendaciones para el socio (dni): 11111111...

RECOMENDACIONES PARA JUAN PÉREZ
¡Hola, Juan Pérez! Es un placer verte de nuevo por aquí. He estado revisando tu historial de lectura y veo que tienes un gusto muy particular por las narrativas que invitan a la reflexión y los mundos bien construidos, ya sea en la ciencia ficción o en la fantasía.

Basándome en tu aprecio por la complejidad de "Fundación" de Asimov y la originalidad de "Ficciones" de Borges, he seleccionado tres títulos de nuestra colección que creo que podrían interesarte mucho:

TÍTULO: Rayuela
PORTADA: Una silueta abstracta de una figura humana saltando sobre un tablero de ajedrez o un laberinto de calles parisinas, con tonos sepia y un toque de azul melancólico. Las letras del título, en una tipografía elegante pero ligeramente desordenada, parecen flotar o caer, sugiriendo la naturaleza fragmentada y lúdica de la obra.
JUSTIFICACIÓN: Juan, viendo que disfrutó de la profundidad y la exper